# Lecture 3.1 — Function Tools: The `@function_tool` Decorator

**Course:** OpenAI Agents SDK — Complete Course  
**Section:** 03 — Tools: Extending Agent Capabilities  

In this notebook, you will learn how to convert ordinary Python functions into agent-callable tools using the `@function_tool` decorator. You will explore both usage forms of the decorator, inspect the JSON schema it generates, understand how sync and async tools differ in execution, and see how advanced parameters like `is_enabled`, `strict_mode`, and Pydantic `Field` constraints work in practice.

## Cell 1 — Install the SDK

📌 **Notebook update notice:** this lecture's markdown references `openai-agents==0.17.7` as the pinned version. Since recording, a downstream dependency change (`openai>=2.45.0`, released July 9, 2026) broke `openai-agents` versions below 0.18.1 — `Runner.run()` will fail on the version stated above. This notebook has been updated to pin `openai-agents==0.18.3`, which fixes the issue without changing any of the code or concepts taught in the lecture. Please use the version pinned below, not the one mentioned in the recording.

The cell below installs the `openai-agents` package, pinned to version **0.18.3** for reproducibility. Pinning ensures that every example in this notebook runs against the same API surface that was tested when the course was written.

If you want the latest version instead, run:

In [ ]:
# Pinned for reproducibility. Updated after recording — see the
# notice above. Originally pinned to 0.17.7 as stated in the
# video; updated to 0.18.3 to fix a breaking change introduced
# by openai>=2.45.0 (July 9, 2026).
# To use the latest version instead, run: pip install openai-agents
!pip install openai-agents==0.18.3 -q

## 2. API Key Setup

This notebook uses the **Google Colab Secrets** method to load your OpenAI API key securely — no key is ever hardcoded in the notebook.

**Steps to add your key in Colab:**
1. Click the 🔑 **Secrets** icon in the left sidebar (or go to **Tools → Secrets**).
2. Click **Add new secret**.
3. Set **Name** to `OPENAI_API_KEY`.
4. Paste your OpenAI API key as the **Value**.
5. Toggle **Notebook access** to ON.
6. Run the cell below.

**Running locally?** Set the environment variable in your terminal before launching Jupyter:
```bash
export OPENAI_API_KEY="your-key-here"
```
Then comment out the `userdata` lines below and the key will be picked up automatically.

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 3. Model Name Variable

We declare a single `MODEL_NAME` variable here and use it everywhere an `Agent` is defined. This means you can switch to a different model by changing one line — no hunting through the notebook.

See the latest available models at: https://platform.openai.com/docs/models

In [ ]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## 4. Imports

Here is everything this notebook requires:

| Import | Why it is here |
|---|---|
| `asyncio` | Used in the async tool example (simulating I/O with `asyncio.sleep`) |
| `Annotated` | Enables `Annotated[type, Field(...)]` syntax for Pydantic constraints |
| `Any` | Used in the `is_enabled` callback signature |
| `Field` | Pydantic `Field` for adding constraints and descriptions to parameters |
| `Agent` | The core agent class |
| `ModelSettings` | Controls model-level parameters like temperature and reasoning |
| `RunContextWrapper` | Wraps the run-level context object — imported here for the `is_enabled` demo. Full context usage is covered in Lecture 3.5. |
| `Runner` | Executes the agent loop. We always use `await Runner.run()` in Jupyter/Colab. |
| `function_tool` | The decorator that converts a Python function into a `FunctionTool`. |

> **Note on `RunContextWrapper`:** It is imported here because the `is_enabled` demo passes a callable that receives `(run_context: RunContextWrapper, agent)`. The full story of how context flows through tools is Lecture 3.5.

In [ ]:
import asyncio
from typing import Annotated, Any

from pydantic import Field

from agents import (
    Agent,
    ModelSettings,
    RunContextWrapper,
    Runner,
    function_tool,
)

## 5. What `@function_tool` Does

The `@function_tool` decorator converts a plain Python function into a `FunctionTool` object that an `Agent` can call. It automates three things that would otherwise require manual wiring:

### Three Automatic Behaviours

1. **Signature → JSON Schema.** The decorator parses your function's type annotations using Python's `inspect` module and builds a JSON schema for the tool's parameters. This schema is what the model sees when it decides whether and how to call the tool.

2. **Docstring → Tool Description.** The first paragraph of your docstring becomes the tool's description — the text the model uses to understand what the tool does. Docstring styles **Google**, **Sphinx**, and **NumPy** are all supported and auto-detected (you can also force a style with `docstring_style=`).

3. **Docstring Args Section → Parameter Descriptions.** The `Args:` block in your docstring becomes per-parameter descriptions in the JSON schema, helping the model understand what each argument means.

### Two Usage Forms

```python
# Form 1: no parentheses — uses all defaults
@function_tool
def my_tool(x: int) -> str: ...

# Form 2: with parentheses — customise parameters
@function_tool(name_override="my_custom_name", strict_mode=False)
def my_tool(x: int) -> str: ...
```

Both forms produce the same `FunctionTool` object. Pick the form that matches your needs.

### Sync vs Async

- **Sync functions** are automatically run in `asyncio.to_thread()` — they never block the event loop.
- **Async functions** are awaited directly by the SDK.
- Both work without any special handling in Jupyter or Colab.

### The Context Parameter

If the **first** parameter of your function is `RunContextWrapper` (or `ToolContext`), the SDK excludes it from the JSON schema and passes the run context automatically. You never pass it yourself. This is covered in full in **Lecture 3.5**.

## 6. Simplest Form — `@function_tool` with No Parentheses

When you write `@function_tool` with no parentheses, the decorator is applied directly with all default settings. This is the most concise form and is the right choice when you don't need to customise anything.

After decorating, the function is replaced by a `FunctionTool` object. We print three of its properties to make the schema generation concrete:

| Property | What it shows |
|---|---|
| `.name` | The tool name the model sees (from the function name by default) |
| `.description` | The tool description (from the first docstring paragraph) |
| `.params_json_schema` | The exact JSON schema sent to the model — inspect this when debugging |

> **Tip:** Always print `.params_json_schema` when a tool isn't being called as expected. What you see there is exactly what the model receives.

In [ ]:
@function_tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city.

    Args:
        city: The name of the city to get weather for.
    """
    return f"The weather in {city} is sunny and 24\u00b0C."


print("Tool name:", get_weather.name)
print("Tool description:", get_weather.description)
print("Tool schema:", get_weather.params_json_schema)

Tool name: get_weather
Tool description: Returns the current weather for a given city.
Tool schema: {'properties': {'city': {'description': 'The name of the city to get weather for.', 'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weather_args', 'type': 'object', 'additionalProperties': False}


## 7. Run an Agent with the Tool

We now wire `get_weather` to an `Agent` by adding it to the `tools` list. The model receives the JSON schema and decides when to call the tool — we don't call it ourselves.

A few things to note about the agent configuration:

- **`model=MODEL_NAME`** — we always use the variable declared earlier, never a hardcoded string.
- **`tools=[get_weather]`** — the `FunctionTool` object goes directly in the list.
- **`await Runner.run(...)`** — always `await` in Jupyter/Colab. `Runner.run_sync()` raises a `RuntimeError` in environments that already have an event loop.

In [ ]:
agent = Agent(
    name="Weather Agent",
    instructions=(
        "You are a helpful weather assistant. "
        "Use the get_weather tool to answer weather questions."
    ),
    model=MODEL_NAME,
    tools=[get_weather],
)

result = await Runner.run(agent, "What is the weather in Tokyo?")
print(result.final_output)

Tokyo is sunny and 24°C.


## 8. Async Function Tool

The `@function_tool` decorator works equally well on `async def` functions. The SDK awaits them directly, making async tools the right choice for any I/O-bound operation: HTTP calls, database queries, file reads, etc.

For comparison:

| Tool type | How the SDK runs it | When to use |
|---|---|---|
| `def` (sync) | `asyncio.to_thread()` — runs in a thread pool | CPU-bound or simple logic |
| `async def` | `await` — runs on the event loop | I/O-bound: APIs, DBs, files |

Both approaches are fully compatible with Jupyter and Colab — no special handling required.

In [ ]:
@function_tool
async def get_stock_price(ticker: str) -> str:
    """Returns the current stock price for a ticker symbol.

    Args:
        ticker: The stock ticker symbol, e.g. AAPL, GOOG.
    """
    await asyncio.sleep(0.1)  # Simulating an async API call
    return f"The current price of {ticker} is $142.50."


print("Async tool name:", get_stock_price.name)
print("Schema:", get_stock_price.params_json_schema)

Async tool name: get_stock_price
Schema: {'properties': {'ticker': {'description': 'The stock ticker symbol, e.g. AAPL, GOOG.', 'title': 'Ticker', 'type': 'string'}}, 'required': ['ticker'], 'title': 'get_stock_price_args', 'type': 'object', 'additionalProperties': False}


## 9. `name_override` and `description_override`

By default, the tool name comes from the function name and the description from the docstring. You can override both with decorator parameters:

| Parameter | Effect |
|---|---|
| `name_override` | Changes the tool name the model sees — the Python function name is unchanged |
| `description_override` | Replaces the docstring description entirely — the docstring is ignored |

When are these useful?
- **`name_override`:** When your internal function name (`get_temperature`) doesn't make a good tool name for the model (`fetch_current_temperature`).
- **`description_override`:** When you want a precise, model-facing description that differs from your developer-facing docstring, or when you want to be explicit rather than relying on docstring parsing.

In [ ]:
@function_tool(
    name_override="fetch_current_temperature",
    description_override=(
        "Fetches the current temperature in Celsius for a "
        "given city. Use this when the user asks specifically "
        "about temperature."
    ),
)
def get_temperature(city: str) -> str:
    """Internal implementation — docstring ignored."""
    return f"The current temperature in {city} is 22\u00b0C."


print("Tool name:", get_temperature.name)
print("Description:", get_temperature.description)

Tool name: fetch_current_temperature
Description: Fetches the current temperature in Celsius for a given city. Use this when the user asks specifically about temperature.


## 10. `use_docstring_info=False`

Setting `use_docstring_info=False` disables all docstring parsing. The tool will have:
- **Name:** still from the function name (or `name_override` if set)
- **Description:** empty string
- **Parameter descriptions:** empty

This is useful when you are providing `description_override` and you explicitly do not want any content from the docstring to bleed through. Without this flag, if you set `description_override` but leave `use_docstring_info=True`, the docstring args section could still contribute parameter descriptions to the schema.

After running the cell, notice that `description` is an empty string — the docstring was completely ignored.

In [ ]:
@function_tool(use_docstring_info=False)
def compute_sum(a: int, b: int) -> int:
    """This docstring will be completely ignored."""
    return a + b


print("Description:", repr(compute_sum.description))
print("Schema:", compute_sum.params_json_schema)

Description: ''
Schema: {'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'compute_sum_args', 'type': 'object', 'additionalProperties': False}


## 11. Pydantic `Field` Constraints

You can attach Pydantic `Field` objects to parameters using `Annotated`. The constraints are included in the JSON schema sent to the model, and Pydantic validates the model's JSON input against them *before* your function is even called.

Two supported forms:

```python
# Annotated form (preferred — clean and explicit)
arg: Annotated[int, Field(ge=0, le=100, description="Score from 0 to 100")]

# Default-based form
arg: int = Field(..., ge=0, le=100)
```

Common `Field` constraints and what they mean:

| Constraint | Meaning |
|---|---|
| `ge=0` | Greater than or equal to 0 |
| `le=100` | Less than or equal to 100 |
| `gt=0` | Strictly greater than 0 |
| `lt=100` | Strictly less than 100 |
| `min_length=1` | String must have at least 1 character |
| `description="..."` | Per-parameter description in the schema |

After running, inspect the schema to see how `ge` and `le` appear as `minimum` and `maximum` in the JSON schema.

In [ ]:
@function_tool
def score_submission(
    score: Annotated[
        int,
        Field(ge=0, le=100, description="Score from 0 to 100"),
    ],
    category: Annotated[
        str,
        Field(description="Category of the submission"),
    ],
) -> str:
    """Records a submission score.

    Args:
        score: The numeric score value.
        category: The submission category.
    """
    return f"Recorded score {score} in category '{category}'."


print("Schema:", score_submission.params_json_schema)

Schema: {'properties': {'score': {'description': 'The numeric score value.', 'maximum': 100, 'minimum': 0, 'title': 'Score', 'type': 'integer'}, 'category': {'description': 'The submission category.', 'title': 'Category', 'type': 'string'}}, 'required': ['score', 'category'], 'title': 'score_submission_args', 'type': 'object', 'additionalProperties': False}


## 12. `strict_mode` — How It Shapes the Tool Definition Sent to the Model

`strict_mode` is not just a validation flag. It changes two things **at decoration time**, before any agent is created or any run is started.

### 1. The parameters schema structure

With `strict_mode=True` (the default), the SDK builds a fully constrained schema. Every parameter is added to `required` and `additionalProperties` is set to `false`:

```json
{
  "type": "object",
  "properties": {
    "query": { "type": "string" }
  },
  "required": ["query"],
  "additionalProperties": false
}
```

With `strict_mode=False`, the schema is relaxed — optional parameters stay out of `required` and `additionalProperties` is not constrained:

```json
{
  "type": "object",
  "properties": {
    "query":  { "type": "string" },
    "limit":  { "type": "integer" }
  },
  "required": ["query"]
}
```

### 2. The `strict` field in the full tool definition

The SDK sets `"strict": true` or `"strict": false` in the tool definition it sends to OpenAI's API:

```json
{
  "type": "function_call",
  "name": "flexible_search",
  "description": "Searches a knowledge base with optional filters.",
  "parameters": { "..." : "..." },
  "strict": true
}
```

When `strict: true`, OpenAI activates **constrained decoding** — the model's output is forced to conform to the schema token by token as it generates. It cannot skip a required field or produce malformed JSON. The constraint is enforced at generation time, not just validated after.

When `strict: false`, there is no constrained decoding. The model generates the arguments string freely, guided by the schema as a hint but not bound by it.

### The critical consequence: optional parameters cause a decoration-time error

If your function has optional parameters — defaults, `dict[str, Any]`, `X | None = None` — and you leave `strict_mode=True`, the SDK raises an error the **moment the decorator runs**. Before any agent is created. Before anything is sent to the model:

```python
@function_tool          # strict_mode=True — this ERRORS at decoration time
def flexible_search(
    query: str,
    filters: dict[str, Any] | None = None,  # optional: not allowed in strict
    limit: int = 10,                         # has default: not allowed in strict
) -> str: ...
```

The fix is to explicitly opt out:

```python
@function_tool(strict_mode=False)   # relaxed schema, no constrained decoding
def flexible_search(...): ...
```

### Trade-off

| | strict_mode=True | strict_mode=False |
|---|---|---|
| Schema | All params in `required`, `additionalProperties: false` | Optional params allowed |
| Tool definition | `"strict": true` | `"strict": false` |
| Model output | Constrained decoding — structurally guaranteed | Generated freely — usually correct, occasionally malformed |
| Function signature | All params must be required, no `dict[str, Any]` | Natural Python signatures allowed |

**Rule:** Start with `True`. Only reach for `False` when your function genuinely needs optional parameters or flexible dict types.

In the cell below we print the **full tool definition** — name, description, parameters schema, and strict flag — so you can see exactly what gets sent to the model.

In [ ]:
import json

@function_tool(strict_mode=False)
def flexible_search(
    query: str,
    filters: dict[str, Any] | None = None,
    limit: int = 10,
) -> str:
    """Searches a knowledge base with optional filters.

    Args:
        query: The search query string.
        filters: Optional dict of filter key-value pairs.
        limit: Maximum number of results to return.
    """
    return f"Found 3 results for '{query}' with limit {limit}."


# Print the full tool definition sent to the model:
# name + description + parameters schema + strict flag
tool_definition = {
    "type": "function_call",
    "name": flexible_search.name,
    "description": flexible_search.description,
    "parameters": flexible_search.params_json_schema,
    "strict": flexible_search.strict_json_schema,
}
print(json.dumps(tool_definition, indent=2))

{
  "type": "function_call",
  "name": "flexible_search",
  "description": "Searches a knowledge base with optional filters.",
  "parameters": {
    "properties": {
      "query": {
        "description": "The search query string.",
        "title": "Query",
        "type": "string"
      },
      "filters": {
        "anyOf": [
          {
            "additionalProperties": true,
            "type": "object"
          },
          {
            "type": "null"
          }
        ],
        "default": null,
        "description": "Optional dict of filter key-value pairs.",
        "title": "Filters"
      },
      "limit": {
        "default": 10,
        "description": "Maximum number of results to return.",
        "title": "Limit",
        "type": "integer"
      }
    },
    "required": [
      "query"
    ],
    "title": "flexible_search_args",
    "type": "object"
  },
  "strict": false
}


## 13. `is_enabled` — Dynamic Tool Availability

`is_enabled` is your feature flag system for tools. It can be:
- A **`bool`**: the tool is always visible (`True`) or always hidden (`False`).
- A **callable** with signature `(run_context: RunContextWrapper[Any], agent: Any) -> bool`: the SDK calls it at runtime to decide whether to include the tool in the model's tool list for this particular run.

When `is_enabled` returns `False`, the tool is **not in the model's tool list at all** — it's invisible to the model, not just ignored. This makes it perfect for:
- **Tier-based access control:** Only premium users see the premium tool.
- **Feature flags:** Enable/disable tools based on configuration or runtime state.
- **Conditional availability:** A tool that only makes sense in certain contexts.

The callable receives the `RunContextWrapper` so it can inspect your context object. In this example, we check for a hypothetical `is_premium` attribute on the context.

In [ ]:
def check_premium_access(
    run_context: RunContextWrapper[Any],
    agent: Any,
) -> bool:
    """Returns True only for premium users."""
    return getattr(run_context.context, "is_premium", False)


@function_tool(is_enabled=check_premium_access)
def generate_report(topic: str) -> str:
    """Generates a detailed report on a topic.

    Args:
        topic: The topic to generate a report on.
    """
    return f"Premium report generated for topic: {topic}."


print("Tool name:", generate_report.name)
print("Note: hidden from non-premium users at runtime.")

Tool name: generate_report
Note: hidden from non-premium users at runtime.


## 14. Full `@function_tool` Parameter Reference

All parameters accepted by the `@function_tool` decorator, verified from the SDK source:

| Parameter | Type | Default | Description |
|---|---|---|---|
| `name_override` | `str \| None` | `None` | Override the tool name exposed to the model |
| `description_override` | `str \| None` | `None` | Override the description (replaces docstring) |
| `docstring_style` | `str \| None` | `None` | Force docstring style: `"google"`, `"sphinx"`, `"numpy"` |
| `use_docstring_info` | `bool` | `True` | Parse docstring for description and arg descriptions |
| `failure_error_function` | callable \| None | default handler | Error handler when the tool raises an exception. Default sends error message to the LLM. Pass `None` to re-raise instead. |
| `strict_mode` | `bool` | `True` | Sets `"strict": true` in the tool definition and forces all params into `required` with `additionalProperties: false`. Activates OpenAI constrained decoding — the model cannot skip a field or produce malformed JSON. Optional params or `dict[str, Any]` cause a decoration-time error when `True`. Use `False` only when your function genuinely needs optional params. |
| `is_enabled` | `bool \| callable` | `True` | When `False` or callable returns `False`, tool is hidden from the model at runtime. |
| `needs_approval` | `bool \| callable` | `False` | Pauses the run for human approval before the tool executes. Covered fully in **Update Section U3**. |
| `tool_input_guardrails` | `list \| None` | `None` | Guardrails on tool inputs. Covered in **Update Section U3**. |
| `tool_output_guardrails` | `list \| None` | `None` | Guardrails on tool outputs. Covered in **Update Section U3**. |
| `timeout` | `float \| None` | `None` | Per-call timeout in seconds. Async tools only. |
| `timeout_behavior` | `str` | `"error_as_result"` | `"error_as_result"`: sends timeout message to LLM. `"raise_exception"`: raises `ToolTimeoutError`. |
| `timeout_error_function` | callable \| None | `None` | Custom timeout message formatter. |
| `defer_loading` | `bool` | `False` | Hides tool until the Responses API tool search loads it. |

> **What we haven't covered in this lecture:** `failure_error_function` and `timeout`/`timeout_behavior` in depth (Lecture 3.7), and `needs_approval`, `tool_input_guardrails`, `tool_output_guardrails` (Update Section U3).